### Apple Watch Health Analytics
The purpose of this study is to use the following two datasets to generate insights on a simulated experimental study. Participants were asked to wear an Apple Watch and record several aspects of their activity and health at three different timepoints. Some aspects of the participants changed over the course of the trial, e.g. smoking status. These aspects were recorded in the first dataset, **health_irreg_rhythm**. 

Participants were also organized into cohorts: groups of individuals to help facilitate the study. Each cohort’s average temperature was aggregated for each month of the study and is stored in a second dataset, **health_cohort_monthly**.

The detailed schemas of the two tables are as follows:

**health_irreg_rhythm**
- id: A unique identifier for the participant
- date: Date participant recorded their data
- smoker: "yes" if the person smokes, "no" if they don't
- rings_closed: Median number of rings closed per day
- watch_type: The material type for the Apple Watch
- diff_exer_types: Median of different types of exercise types per week
- max_hr: Maximum heart rate recorded during the week
- irreg_rhythm: "1" if the person exhibited irregular heart rhythm; "0" if they did not.
- cohort: The group of the participant in the study

**health_cohort_monthly**
- cohort: The group of the participant in the study
- month: Month of the study
- mean_temp: The average temperature of the cohort during the specified month




In [0]:
-- Set current and check catalog and schema
USE CATALOG workspace;
USE SCHEMA linkprojects_apple_watch;

SELECT current_catalog(), current_schema();

In [0]:
-- Explore the schema
DESCRIBE SCHEMA EXTENDED linkprojects_apple_watch;

In [0]:
-- Check for any existing volumes.
SHOW VOLUMES

In [0]:
-- List the files inside the volume.
LIST '/Volumes/workspace/linkprojects_apple_watch/rawdata'

In [0]:
-- Preview the CSV data without creating a table. 
SELECT * 
FROM read_files('/Volumes/workspace/linkprojects_apple_watch/rawdata/health_irreg_rhythm.csv')
LIMIT 10;

In [0]:
-- Create a bronze Delta table from the CSV
CREATE TABLE IF NOT EXISTS health_irreg_rhythm_bronze AS
SELECT id, date, smoker, rings_closed, watch_type, diff_exer_types, max_hr, irreg_rhythm, cohort
FROM read_files('/Volumes/workspace/linkprojects_apple_watch/rawdata/health_irreg_rhythm.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
-- Verify the bronze table created
SELECT *
FROM health_irreg_rhythm_bronze;

In the first glimpse of the bronze table, we can identify some issues with the data in the columns `date`, `smoker`, `rings_closed`, and `max_hr`. Let's fix them and create a silver table.

1. The column `date` is a string, but it is actually a date. We need to convert it to a date.
2. The column `smoker` is a string, but it is actually a boolean. We need to convert it to a boolean.
3. The column `rings_closed` has some invalid inputs. Rings closed 4 is highly likely a typo since Apple Watch only has 3 rings to close. We need to replace it with the number 3.
4. The column `max_hr` has some irregular outliers. Maximum heart rate 1600 is not physiologically possible. We need to replace it with the median of the column.

In [0]:
-- Create a silver Delta table from the CSV
CREATE VIEW IF NOT EXISTS health_irreg_rhythm_silver AS
SELECT *,
  CASE
    WHEN date RLIKE '^[A-Za-z]{3}-[0-9]{1,2}-[0-9]{4}$' THEN to_date(date, 'MMM-d-yyyy')
    WHEN date RLIKE '^[0-9]{1,2}/[0-9]{1,2}/[0-9]{2}$' THEN to_date(date, 'M/d/yy')
    ELSE NULL
  END AS date_fixed,
  CASE
    WHEN smoker = 'yes' THEN True
    WHEN smoker IN ('no', 'n') THEN False
    ELSE NULL
  END AS smoker_fixed,
  CASE
    WHEN max_hr = 1600 THEN (SELECT MEDIAN(max_hr) FROM workspace.linkprojects_apple_watch.health_irreg_rhythm_bronze WHERE max_hr != 1600)
    ELSE max_hr
  END AS max_hr_fixed,
  CASE
    WHEN rings_closed = 4 THEN 3
    ELSE rings_closed
  END AS rings_closed_fixed
FROM workspace.linkprojects_apple_watch.health_irreg_rhythm_bronze;

We further investigate the abnormality in the data not obvious from the first glance. Participants were organized into 5 cohorts in this simulated experimental study. However, the total count of cohort 3 and 4 do not match.

In [0]:
-- Verify the total entries in each cohort
SELECT cohort, COUNT(id)
FROM health_irreg_rhythm_silver
GROUP BY cohort;

We further investigate the abnormality in the data not obvious from the first glance. We find that the participant with ID 3 has 1 entry in the cohort 3 and 2 entries in the cohort 4. This is likely a data entry error since each participant should have only 3 entries in the same cohort.

In [0]:
-- Verify the entries of each participant in each cohort
SELECT id, cohort, COUNT(id)
FROM health_irreg_rhythm_silver
GROUP BY cohort, id
HAVING COUNT(id) != 3
Order BY cohort, id;

In [0]:
-- Modify the silver table
CREATE OR REPLACE VIEW health_irreg_rhythm_silver_fixed AS
(
SELECT id, date_fixed, smoker_fixed, rings_closed_fixed, watch_type, diff_exer_types, max_hr_fixed, irreg_rhythm,
  CASE
    WHEN cohort = 3 AND Id = 3 THEN 4
    ELSE cohort
  END AS cohort_fixed
FROM workspace.linkprojects_apple_watch.health_irreg_rhythm_silver
)

In [0]:
SELECT *
FROM health_irreg_rhythm_silver_fixed
LIMIT 10;

Now we have a clean and standardized silver Delta table. We can further generate a refined gold table ready for the downstream analysis for specific business use cases. Let's say we are interested in the smoking status of the participants. Specifically, the number and percent of all participants who:

1. Did not change their smoking status during the trial
2. Started smoking during the trial
3. Quit smoling during the trial
4. Started smoking, but then quit by the end of the trial
5. Quit smoking, but then restarted smoking by the end of the trial

We will generate a gold table which includes aggregated, business-ready data built to answer these specific questions.

In [0]:
-- Create a gold table focusing on smoking status
CREATE OR REPLACE TABLE smoking_categorized_gold AS
WITH smoking_timeline AS (
    SELECT 
        id,
        date_fixed,
        smoker_fixed,
        ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_fixed) as timepoint,
        COUNT(*) OVER (PARTITION BY id) as total_timepoints
    FROM health_irreg_rhythm_silver_fixed
),
smoking_summary AS (
    SELECT 
        id,
        total_timepoints,
        MAX(CASE WHEN timepoint = 1 THEN smoker_fixed END) as first_status,
        MAX(CASE WHEN timepoint = 2 THEN smoker_fixed END) as middle_status,
        MAX(CASE WHEN timepoint = total_timepoints THEN smoker_fixed END) as last_status,
        SUM(CASE WHEN smoker_fixed = true THEN 1 ELSE 0 END) as times_smoked
    FROM smoking_timeline
    GROUP BY id, total_timepoints
)
SELECT 
    id,
    CASE 
        -- 1. Did not change
        WHEN times_smoked = 0 OR times_smoked = total_timepoints 
            THEN 'Did Not Change'
        -- 2. Started smoking
        WHEN first_status = false AND last_status = true
            THEN 'Started Smoking'
        -- 3. Quit smoking
        WHEN first_status = true AND last_status = false
            THEN 'Quit Smoking'
        -- 4. Started then quit
        WHEN first_status = false AND middle_status = true AND last_status = false
            THEN 'Started Then Quit'
        -- 5. Quit then restarted
        WHEN first_status = true AND middle_status = false AND last_status = true
            THEN 'Quit Then Restarted'
        ELSE 'Other'
    END as smoking_category
FROM smoking_summary;

In [0]:
SELECT * 
FROM smoking_categorized_gold
LIMIT 10;

In [0]:
-- Write a query to answer the questions
SELECT 
    smoking_category,
    COUNT(*) as participant_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(DISTINCT id) FROM smoking_categorized_gold), 1) 
        as percent_of_participants
FROM smoking_categorized_gold
GROUP BY smoking_category;